# 03 · Approach 2: Direct Image Embeddings (Voyage AI)

**Strategy:** Embed each comic page image directly using [Voyage AI's `voyage-multimodal-3`](https://docs.voyageai.com/docs/multimodal-embeddings) — no text description needed. Both images and text queries live in the same 1024-dimensional embedding space, enabling cross-modal search.

**Namespace:** `ns-image-clip`

**Blog post insight:** CLIP-style embeddings capture *visual* information that text descriptions miss: art style, panel composition, colour tone, action pose. But they're less precise for abstract semantic concepts — a query for *"betrayal"* works better with a language model.

---

## When to use this approach
- You want visual similarity search (find pages that look alike)
- You need cross-modal retrieval: text queries returning images, or image queries returning images
- You have no text transcripts and don't want to generate them

In [ ]:
import json
import os
import time
from pathlib import Path

import voyageai
from dotenv import load_dotenv
from pinecone import Pinecone
from PIL import Image
from tqdm import tqdm

load_dotenv()

TRANSCRIPTS_DIR = Path("data/transcripts")
INDEX_NAME      = "comics-multimodal"
NAMESPACE       = "ns-image-clip"
BATCH_SIZE      = 10   # Voyage multimodal batches are smaller due to image payload

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
vo = voyageai.Client()  # reads VOYAGE_API_KEY from env

# Connect to the shared index (created in notebook 02)
info = pc.preview.indexes.describe(name=INDEX_NAME)
INDEX_HOST = info.host
index = pc.index(host=INDEX_HOST)
print(f"Connected to {INDEX_NAME} at {INDEX_HOST}")

## Embed page images with Voyage AI

Each page is embedded as a PIL Image. Voyage `voyage-multimodal-3` accepts images directly — no base64 encoding step needed.

**Cost:** ~$0.000006/image. For 150 pages ≈ $0.0009 total.

In [ ]:
records = [json.loads(p.read_text()) for p in sorted(TRANSCRIPTS_DIR.glob("*.json"))]
print(f"Pages to embed: {len(records)}")

In [ ]:
def embed_images_batch(paths: list[str]) -> list[list[float]]:
    """Embed a batch of image paths. Returns list of 1024-dim vectors."""
    inputs = [[Image.open(p).convert("RGB")] for p in paths]
    result = vo.multimodal_embed(inputs, model="voyage-multimodal-3", input_type="document")
    return result.embeddings


# Embed all pages (cache vectors in the JSON records to avoid re-billing)
for i in tqdm(range(0, len(records), BATCH_SIZE), desc="Embedding images"):
    batch = records[i : i + BATCH_SIZE]
    # Skip pages that already have a cached vector
    needs_embedding = [r for r in batch if not r.get("image_vector")]
    if not needs_embedding:
        continue
    paths = [r["file_path"] for r in needs_embedding]
    vectors = embed_images_batch(paths)
    for r, v in zip(needs_embedding, vectors):
        r["image_vector"] = v
        path = TRANSCRIPTS_DIR / f"{r['page_id']}.json"
        path.write_text(json.dumps(r, ensure_ascii=False, indent=2))

print("Image vectors ready.")

## Upsert to Pinecone (ns-image-clip)

Unlike the text-caption namespace, here we **bring our own vectors** — we pass the 1024-dim Voyage vector directly rather than letting Pinecone's integrated inference embed a text field.

In [ ]:
documents = [
    {
        "_id":        r["page_id"],
        "embedding":  r["image_vector"],   # pre-computed Voyage vector (BYOV)
        "dialogue":   r["all_dialogue"],
        "comic_title": r["comic_title"],
        "page_id":    r["page_id"],
        "page_num":   r["page_num"],
        "file_path":  r["file_path"],
    }
    for r in records
]

index.documents.batch_upsert(
    namespace=NAMESPACE,
    documents=documents,
    batch_size=BATCH_SIZE,
    show_progress=True,
)
print(f"Upserted {len(documents)} pages to {NAMESPACE}.")

## Search demos

All queries are embedded with Voyage AI at query time — both text and image queries go through the same model, landing in the same vector space.

In [ ]:
from IPython.display import display


def show_results(matches, label=""):
    if label:
        print(f"\n{'='*60}\n{label}\n{'='*60}")
    for m in matches:
        comic = getattr(m, "comic_title", "")
        page  = getattr(m, "page_num", "")
        print(f"[{m._score:.3f}] {m._id}  ({comic}, p{page})")
        fp = getattr(m, "file_path", None)
        if fp and Path(fp).exists():
            img = Image.open(fp)
            img.thumbnail((250, 350))
            display(img)

In [ ]:
# Text-to-image: embed a text query with Voyage, search image vectors
text_query = "dark alley chase scene at night"
result = vo.multimodal_embed([[text_query]], model="voyage-multimodal-3", input_type="query")
query_vector = result.embeddings[0]

resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": query_vector}],
    include_fields=["page_id", "comic_title", "page_num", "file_path"],
)
show_results(resp.matches, f"Text-to-image: '{text_query}'")

In [ ]:
# Image-to-image: use a page image as the query to find visually similar pages
query_page = records[0]  # change to any page you want to use as a query
query_img  = Image.open(query_page["file_path"]).convert("RGB")
result = vo.multimodal_embed([[query_img]], model="voyage-multimodal-3", input_type="query")
img_query_vector = result.embeddings[0]

print(f"Query image: {query_page['page_id']}")
query_img.thumbnail((250, 350))
display(query_img)

resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=4,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": img_query_vector}],
    # Exclude the query page itself
    filter={"page_id": {"$ne": query_page["page_id"]}},
    include_fields=["page_id", "comic_title", "page_num", "file_path"],
)
show_results(resp.matches, "Image-to-image: visually similar pages")

In [ ]:
# Cross-comic visual style: find action scenes across all comics
result = vo.multimodal_embed([["superhero punching villain action scene"]], model="voyage-multimodal-3", input_type="query")
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": result.embeddings[0]}],
    include_fields=["page_id", "comic_title", "page_num", "file_path"],
)
show_results(resp.matches, "Cross-comic: 'superhero punching villain action scene'")